# Regularisation and tuning on the offline plus online augmentation arm (ST-GCN)

Continues `paper_premise_test.ipynb`. The arm `offline_basic_online_advanced` (classes built up to 500 with fixed synthetic clips, then the
advanced online bundle on top) was the best of the three premise arms on its single run. This notebook asks whether a few
regularisation and optimisation changes make it better, **using validation only** (tuning seeds 77 and 88, the test set is never evaluated).

**Why these changes.** That arm reaches 98.6% training accuracy against about 76% validation, so the question is generalisation. Each
change is one factor against the control, as in the earlier stages:

| arm | change from the control |
|---|---|
| `control` | the premise arm as it was (recipe from `stgcn_recipe.json`, 9 blocks, `body_court`, SGD lr 0.01, weight decay 5e-3) |
| `tcn_drop_0.1`, `tcn_drop_0.2` | dropout inside the temporal convolutions (was harmful under the old schedule, so retested) |
| `half_width` | first block 32 channels instead of 64 (a quarter of the parameters), a direct capacity cut |
| `ema_0.999` | evaluate and select an exponential moving average of the weights instead of the live weights |
| `lr_0.005`, `lr_0.02` | learning rate halved and doubled |
| `wd_0.01` | weight decay doubled (it was tuned before the offline build-up existed) |

**Selection metric and rule, fixed before running.** Both top-1 and MCA matter here, so the checkpoint inside each run is chosen on the
**composite**, the mean of validation top-1 and validation MCA, and arms are compared on the same composite. An arm passes if its mean composite
over the two seeds is at least **0.75 points above the control**, validation top-1 does not fall more than 1.0 below the control, and validation
MCA does not fall more than 1.5 below it. The best passing arm of each family (dropout, width, EMA, learning rate, weight decay) is then
combined into one `combined` arm, which must pass the same rule against the control. The final choice is the passing arm with the highest
composite (ties within 0.3 prefer fewer changes), or the control if nothing passes.

**Noise.** With two seeds and 781 validation clips, composite differences under about 1 point are within noise, and validation MCA has 11 to 13 clips for
seven classes. A pass is a reason to run reporting seeds on the test set, not a result in itself. Changes to `training.py` (EMA, composite
metric) and `models/stgcn.py` (`base_channels`) are backwards compatible.

Logs to `BadmintonPoseAnalysis/experiments/premise_regularisation`.

In [1]:
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

import json
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from clearml import Task

REPO = os.environ.get("BADMINTON_REPO", "/workspace/badminton-honours-project")
DATA = os.environ.get("BADMINTON_DATA", "/workspace/data")
OUTPUTS = os.environ.get("BADMINTON_OUTPUTS", "/workspace/outputs")
sys.path.insert(0, REPO)
import dataset as ds
from models import stgcn as stgcn_model
import training as tr

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(torch.__version__, device, torch.cuda.get_device_name(0) if device.type == "cuda" else "")
plt.rcParams.update({"figure.dpi": 110, "font.size": 9, "axes.spines.top": False, "axes.spines.right": False})

2.1.0+cu118 cuda NVIDIA GeForce RTX 4090


In [2]:
PROJECT = "BadmintonPoseAnalysis/experiments/premise_regularisation"
KEYPOINTS = f"{DATA}/keypoints_interp"
OFFLINE_ROOT = f"{DATA}/keypoints_offline_aug"
OUT = f"{OUTPUTS}/experiments/premise_regularisation"
os.makedirs(OUT, exist_ok=True)

SEEDS = [77, 88]
SPLITS = ("train", "val")                 # the test set is never evaluated here
NORMALISATION, USE_COURT = "body", True
NUM_WORKERS, EVAL_WORKERS = int(os.environ.get("NUM_WORKERS", 12)), 8
SELECT_METRIC = "composite"               # mean of validation top1 and MCA
OFFLINE_TARGET = 500
MIN_GAIN, MAX_TOP1_DROP, MAX_MCA_DROP, TIE = 0.0075, 0.010, 0.015, 0.003

with open(f"{REPO}/configs/stgcn_recipe.json") as f:
    RECIPE = json.load(f)

BASE = {"options": {}, "base_channels": 64, "lr": RECIPE["lr"], "weight_decay": RECIPE["weight_decay"], "ema_decay": None}
ARMS = {
    "control": {},
    "tcn_drop_0.1": {"options": {"tcn_dropout": 0.1}},
    "tcn_drop_0.2": {"options": {"tcn_dropout": 0.2}},
    "half_width": {"base_channels": 32},
    "ema_0.999": {"ema_decay": 0.999},
    "lr_0.005": {"lr": 0.005},
    "lr_0.02": {"lr": 0.02},
    "wd_0.01": {"weight_decay": 0.01},
}
FAMILIES = {"dropout": ["tcn_drop_0.1", "tcn_drop_0.2"], "width": ["half_width"], "ema": ["ema_0.999"],
            "lr": ["lr_0.005", "lr_0.02"], "weight_decay": ["wd_0.01"]}


def spec_of(name):
    return {**BASE, **ARMS[name]}


rng = np.random.RandomState(0)
print("recipe:", RECIPE)

recipe: {'model': 'ST-GCN', 'normalisation': 'body_court', 'selected': 'combination', 'changes_from_baseline': {'weight_decay': 0.005, 'target_per_class': 300}, 'augmentation': 'advanced', 'min_native_frames': 5, 'batch_size': 16, 'lr': 0.01, 'momentum': 0.9, 'gamma': 0.1, 'select_metric': 'top1', 'weight_decay': 0.005, 'label_smoothing': 0.0, 'schedule': 'step10', 'target_per_class': 300, 'num_epochs': 60, 'patience': 20, 'tuning': {'seeds': [11, 22], 'val_top1': 0.7612, 'baseline_val_top1': 0.7394}}


In [3]:
manifest = ds.load_filtered_manifest(f"{DATA}/split_manifest.csv", f"{DATA}/extraction_log.csv",
                                     f"{DATA}/keypoints_interp_log.csv", RECIPE["min_native_frames"])
class_names = sorted(manifest["class_name"].unique())
class_to_id = {c: i for i, c in enumerate(class_names)}
K = len(class_names)
split_hash = open(f"{DATA}/split_manifest.sha256").read().split()[0]

synthetic = ds.build_offline_augmented_set(manifest, KEYPOINTS, OFFLINE_ROOT, target=OFFLINE_TARGET, seed=0, policy="basic")
manifest_offline = pd.concat([manifest, synthetic], ignore_index=True)
in_channels = stgcn_model.n_channels(USE_COURT)
forward = stgcn_model.make_forward(USE_COURT)
print(f"{len(synthetic)} synthetic clips; training clips {(manifest['split'] == 'train').sum()} -> {(manifest_offline['split'] == 'train').sum()}")

manual review: dropped 4 clips
short clip rule: dropped 4 train clips with fewer than 5 real frames
7810 clips remain (train 6247, test 782, val 781)
3913 synthetic clips; training clips 6247 -> 10160


## Smoke test

Every arm must build and train one step, the half-width model must have fewer parameters and a matching head, and the EMA path
in `train_model` must run for a full epoch and hand back finite weights and a composite column.

In [4]:
class _Stub:
    def report_scalar(self, *a, **k):
        pass


tr.seed_everything(0)
loaders = tr.build_loaders(manifest_offline, KEYPOINTS, class_to_id, "advanced", NORMALISATION, seed=0,
                           batch_size=RECIPE["batch_size"], target_per_class=RECIPE["target_per_class"],
                           num_workers=2, eval_workers=2, balanced=False)
batch = next(iter(loaders["train"]))
rows = []
for name in ARMS:
    spec = spec_of(name)
    model = stgcn_model.build_stgcn(in_channels, K, base_channels=spec["base_channels"], **spec["options"]).to(device)
    logits = forward(model, batch, device)
    assert logits.shape == (RECIPE["batch_size"], K)
    nn.CrossEntropyLoss()(logits, batch["label"].to(device)).backward()
    assert not [n for n, p in model.named_parameters() if p.requires_grad and p.grad is None], name
    rows.append({"arm": name, "parameters": sum(p.numel() for p in model.parameters()), "lr": spec["lr"],
                 "weight decay": spec["weight_decay"], "ema": spec["ema_decay"], "options": str(spec["options"] or "")})
    del model
table = pd.DataFrame(rows)
print(table.to_string(index=False))
assert table.set_index("arm").loc["half_width", "parameters"] < 0.4 * table.set_index("arm").loc["control", "parameters"]

# one real epoch through train_model with EMA on
tr.seed_everything(0)
model = stgcn_model.build_stgcn(in_channels, K).to(device)
opt = torch.optim.SGD(model.parameters(), lr=RECIPE["lr"], momentum=RECIPE["momentum"], weight_decay=RECIPE["weight_decay"])
sched = torch.optim.lr_scheduler.StepLR(opt, step_size=10, gamma=RECIPE["gamma"])
best_state, hist, best_epoch = tr.train_model(model, forward, loaders, opt, sched, nn.CrossEntropyLoss(), device, _Stub(),
                                              num_epochs=1, patience=5, select_metric="composite", splits=SPLITS, ema_decay=0.999)
assert "val_composite" in hist.columns and np.isfinite(hist["val_composite"]).all()
assert all(torch.isfinite(v.float()).all() for v in best_state.values())
print(f"EMA epoch ran: val top1 {hist['val_top1'].iloc[0]:.3f}, mca {hist['val_mca'].iloc[0]:.3f}, composite {hist['val_composite'].iloc[0]:.3f}")
del model, loaders
print("smoke test passed")

         arm  parameters    lr  weight decay   ema              options
     control     3028283 0.010         0.005   NaN                     
tcn_drop_0.1     3028283 0.010         0.005   NaN {'tcn_dropout': 0.1}
tcn_drop_0.2     3028283 0.010         0.005   NaN {'tcn_dropout': 0.2}
  half_width      767579 0.010         0.005   NaN                     
   ema_0.999     3028283 0.010         0.005 0.999                     
    lr_0.005     3028283 0.005         0.005   NaN                     
     lr_0.02     3028283 0.020         0.005   NaN                     
     wd_0.01     3028283 0.010         0.010   NaN                     
epoch   0  loss 2.032  train 0.049  val 0.051 (mca 0.056)  *
EMA epoch ran: val top1 0.051, mca 0.056, composite 0.053
smoke test passed


In [5]:
results_path = f"{OUT}/results.csv"


def load_done():
    return pd.read_csv(results_path) if os.path.exists(results_path) else pd.DataFrame(columns=["config", "seed"])


def run_arm(name, spec, seed):
    done = load_done()
    if ((done["config"] == name) & (done["seed"] == seed)).any():
        print(f"skip {name} seed {seed}, already done")
        return
    run_name = f"reg_{name}_s{seed}"
    print(f"\n=== {run_name}  {spec} ===")
    tr.seed_everything(seed)
    task = Task.init(project_name=PROJECT, task_name=run_name, task_type=Task.TaskTypes.training,
                     tags=["premise_regularisation", name, f"seed{seed}"], reuse_last_task_id=False,
                     auto_connect_frameworks={"matplotlib": False, "pytorch": False})
    task.connect({**RECIPE, **spec, "arm": name, "seed": seed, "select_metric": SELECT_METRIC, "train_set": "offline_basic_online_advanced",
                  "keypoints": KEYPOINTS, "split_sha256": split_hash, "evaluated_splits": list(SPLITS)})
    loaders = tr.build_loaders(manifest_offline, KEYPOINTS, class_to_id, "advanced", NORMALISATION, seed=seed,
                               batch_size=RECIPE["batch_size"], target_per_class=RECIPE["target_per_class"],
                               num_workers=NUM_WORKERS, eval_workers=EVAL_WORKERS, balanced=False)
    model = stgcn_model.build_stgcn(in_channels, K, base_channels=spec["base_channels"], **spec["options"]).to(device)
    optimizer = torch.optim.SGD(model.parameters(), lr=spec["lr"], momentum=RECIPE["momentum"], weight_decay=spec["weight_decay"])
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=RECIPE["gamma"])
    criterion = nn.CrossEntropyLoss(label_smoothing=RECIPE["label_smoothing"])
    _, history, best_epoch = tr.train_model(model, forward, loaders, optimizer, scheduler, criterion, device, task.get_logger(),
                                            num_epochs=RECIPE["num_epochs"], patience=RECIPE.get("patience", 20),
                                            select_metric=SELECT_METRIC, splits=SPLITS, ema_decay=spec["ema_decay"])
    metrics = tr.report_run(task, model, forward, loaders, class_names, device, history, best_epoch,
                            out_dir=f"{OUT}/{run_name}", run_name=run_name, splits=SPLITS, show=False,
                            extra={"config": name, "seed": seed})
    task.close()
    pd.concat([done, pd.DataFrame([metrics])], ignore_index=True).to_csv(results_path, index=False)
    del model, loaders
    torch.cuda.empty_cache()


for name in ARMS:
    for seed in SEEDS:
        run_arm(name, spec_of(name), seed)

skip control seed 77, already done

=== reg_control_s88  {'options': {}, 'base_channels': 64, 'lr': 0.01, 'weight_decay': 0.005, 'ema_decay': None} ===
ClearML Task: created new task id=de6fb4c584264ace8d7693b0976db17f
2026-10-08 06:48:17,735 - clearml.Repository Detection - WARNING - Failed accessing the jupyter server(s): []
ClearML results page: https://app.clear.ml/projects/d08a0665498744dab01bb429a63895b0/tasks/de6fb4c584264ace8d7693b0976db17f/output/log
epoch   0  loss 2.001  train 0.499  val 0.540 (mca 0.470)  *
epoch   1  loss 1.567  train 0.572  val 0.611 (mca 0.553)  *
epoch   3  loss 1.327  train 0.645  val 0.676 (mca 0.531)  *
epoch   4  loss 1.285  train 0.674  val 0.662 (mca 0.571)  *
epoch   5  loss 1.253  train 0.661  val 0.644 (mca 0.587)
epoch   6  loss 1.236  train 0.656  val 0.662 (mca 0.580)  *
epoch   8  loss 1.191  train 0.652  val 0.668 (mca 0.578)  *
epoch   9  loss 1.169  train 0.655  val 0.665 (mca 0.594)  *
epoch  10  loss 0.907  train 0.787  val 0.730 (mca 

███████████████████████████████ 100% | 11.66/11.66 MB [00:03<00:00,  3.01MB/s]: 



=== reg_tcn_drop_0.1_s77  {'options': {'tcn_dropout': 0.1}, 'base_channels': 64, 'lr': 0.01, 'weight_decay': 0.005, 'ema_decay': None} ===


Failed accessing the jupyter server(s): []


ClearML Task: created new task id=d399a365bf18497d8824799c8521bc35
ClearML results page: https://app.clear.ml/projects/d08a0665498744dab01bb429a63895b0/tasks/d399a365bf18497d8824799c8521bc35/output/log
epoch   0  loss 2.059  train 0.507  val 0.579 (mca 0.472)  *
epoch   1  loss 1.594  train 0.548  val 0.575 (mca 0.493)  *
epoch   2  loss 1.469  train 0.599  val 0.635 (mca 0.543)  *
epoch   3  loss 1.366  train 0.623  val 0.654 (mca 0.543)  *
epoch   4  loss 1.308  train 0.642  val 0.680 (mca 0.572)  *
epoch   5  loss 1.301  train 0.635  val 0.652 (mca 0.535)
epoch   8  loss 1.253  train 0.660  val 0.673 (mca 0.595)  *
epoch  10  loss 0.945  train 0.770  val 0.711 (mca 0.620)  *
epoch  11  loss 0.878  train 0.789  val 0.723 (mca 0.638)  *
epoch  13  loss 0.779  train 0.828  val 0.731 (mca 0.646)  *
epoch  15  loss 0.719  train 0.843  val 0.704 (mca 0.618)
epoch  17  loss 0.660  train 0.861  val 0.743 (mca 0.634)  *
epoch  20  loss 0.480  train 0.923  val 0.726 (mca 0.635)
epoch  23  los

...

███████████████████████████████ 100% | 11.66/11.66 MB [00:02<00:00,  4.71MB/s]: 



=== reg_half_width_s77  {'options': {}, 'base_channels': 32, 'lr': 0.01, 'weight_decay': 0.005, 'ema_decay': None} ===


Failed accessing the jupyter server(s): []


ClearML Task: created new task id=0ed6a87755a5416781fa6ed939eb23ec
ClearML results page: https://app.clear.ml/projects/d08a0665498744dab01bb429a63895b0/tasks/0ed6a87755a5416781fa6ed939eb23ec/output/log
epoch   0  loss 2.023  train 0.479  val 0.563 (mca 0.447)  *
epoch   1  loss 1.594  train 0.575  val 0.607 (mca 0.510)  *
epoch   2  loss 1.456  train 0.602  val 0.638 (mca 0.537)  *
epoch   3  loss 1.386  train 0.647  val 0.661 (mca 0.565)  *
epoch   4  loss 1.337  train 0.648  val 0.656 (mca 0.572)  *
epoch   5  loss 1.327  train 0.639  val 0.685 (mca 0.571)  *
epoch  10  loss 0.964  train 0.762  val 0.718 (mca 0.626)  *
epoch  13  loss 0.783  train 0.818  val 0.722 (mca 0.624)  *
epoch  14  loss 0.745  train 0.827  val 0.718 (mca 0.635)  *
epoch  15  loss 0.735  train 0.844  val 0.726 (mca 0.631)  *
epoch  16  loss 0.687  train 0.839  val 0.735 (mca 0.635)  *
epoch  20  loss 0.496  train 0.915  val 0.722 (mca 0.637)
epoch  22  loss 0.425  train 0.935  val 0.735 (mca 0.638)  *
epoch  2

KeyboardInterrupt: 

## Single factor results

In [ ]:
def summarise(names):
    res = load_done()
    g = res[res["config"].isin(names)].groupby("config")
    t = pd.DataFrame({"val top1": g["val_top1"].mean(), "val mca": g["val_mca"].mean(), "val top5": g["val_top5"].mean(),
                      "train val gap": g["train_val_gap"].mean(), "best epoch": g["best_epoch"].mean(),
                      "seeds": g["seed"].count()}).reindex(names)
    t["composite"] = 0.5 * (t["val top1"] + t["val mca"])
    for s in SEEDS:
        r = res[res["seed"] == s].set_index("config")
        t[f"composite seed {s}"] = 0.5 * (r["val_top1"] + r["val_mca"]).reindex(names)
    return t


def judge(t, control="control", n_changes=None):
    base = t.loc[control]
    t["d composite"] = t["composite"] - base["composite"]
    t["d top1"] = t["val top1"] - base["val top1"]
    t["d mca"] = t["val mca"] - base["val mca"]
    t["passes"] = [(n != control) and (r["d composite"] >= MIN_GAIN) and (r["d top1"] >= -MAX_TOP1_DROP) and (r["d mca"] >= -MAX_MCA_DROP)
                   for n, r in t.iterrows()]
    return t


singles = judge(summarise(list(ARMS)))
pct = ["val top1", "val mca", "val top5", "train val gap", "composite", "d composite", "d top1", "d mca"] + [f"composite seed {s}" for s in SEEDS]
show = singles.assign(**{c: (100 * singles[c]).round(2) for c in pct}).round({"best epoch": 1})
show

In [ ]:
winners = {}
for family, names in FAMILIES.items():
    passing = [n for n in names if singles.loc[n, "passes"]]
    if passing:
        winners[family] = max(passing, key=lambda n: singles.loc[n, "composite"])
print("passing arms:", [n for n in ARMS if singles.loc[n, "passes"]] or "none")
print("best passing arm per family:", winners or "none")
COMBINED = {}
if len(winners) >= 2:
    for family, name in winners.items():
        for key, value in ARMS[name].items():
            if key == "options":
                COMBINED.setdefault("options", {}).update(value)
            else:
                COMBINED[key] = value
    print("combined arm:", COMBINED)
else:
    print("fewer than two passing families, no combined arm")

In [ ]:
if COMBINED:
    ARMS["combined"] = COMBINED
    for seed in SEEDS:
        run_arm("combined", spec_of("combined"), seed)

In [ ]:
names = list(ARMS)
final = judge(summarise(names))
n_changes = {n: 0 if n == "control" else (len(winners) if n == "combined" else 1) for n in names}
eligible = ["control"] + [n for n in names if final.loc[n, "passes"]]
best = final.loc[eligible, "composite"].max()
close = [n for n in eligible if best - final.loc[n, "composite"] <= TIE]
selected = min(close, key=lambda n: (n_changes[n], -final.loc[n, "composite"]))
print("passing:", [n for n in names if final.loc[n, "passes"]] or "none")
print("selected:", selected, spec_of(selected))
show = final.assign(**{c: (100 * final[c]).round(2) for c in pct}).round({"best epoch": 1})
show["selected"] = [n == selected for n in show.index]
show

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
x = np.arange(len(names))
ax = axes[0]
ax.bar(x, 100 * final["d composite"], color=["#b0b0b0" if n == "control" else "#4c72b0" for n in names])
for s in SEEDS:
    base_s = final.loc["control", f"composite seed {s}"]
    ax.scatter(x, 100 * (final[f"composite seed {s}"] - base_s), s=18, color="black", zorder=3)
ax.axhline(100 * MIN_GAIN, color="grey", ls="--", lw=1)
ax.axhline(0, color="black", lw=0.8)
ax.set_xticks(x, names, rotation=35, ha="right")
ax.set_ylabel("composite vs control (points)")
ax.set_title("Mean (bar) and each seed (dots); dashed: pass threshold")
ax = axes[1]
ax.scatter(100 * final["val top1"], 100 * final["val mca"], color="#4c72b0")
for n in names:
    ax.annotate(n, (100 * final.loc[n, "val top1"], 100 * final.loc[n, "val mca"]), fontsize=7, xytext=(3, 3), textcoords="offset points")
ax.set_xlabel("validation top-1")
ax.set_ylabel("validation MCA")
ax.set_title("Top-1 against MCA")
fig.tight_layout()
plt.show()

In [ ]:
spec = spec_of(selected)
config = {"stage": "premise_regularisation", "selected": selected, "train_set": "offline_basic_online_advanced",
          "backbone_options": spec["options"], "base_channels": spec["base_channels"], "lr": spec["lr"],
          "weight_decay": spec["weight_decay"], "ema_decay": spec["ema_decay"], "select_metric": SELECT_METRIC,
          "mean_best_epoch": round(float(final.loc[selected, "best epoch"]), 1),
          "tuning": {"seeds": SEEDS, "val_top1": round(float(final.loc[selected, "val top1"]), 4),
                     "val_mca": round(float(final.loc[selected, "val mca"]), 4),
                     "control_val_top1": round(float(final.loc["control", "val top1"]), 4),
                     "control_val_mca": round(float(final.loc["control", "val mca"]), 4)}}
os.makedirs(f"{REPO}/configs", exist_ok=True)
with open(f"{REPO}/configs/premise_regularisation.json", "w") as f:
    json.dump(config, f, indent=2)

task = Task.init(project_name=PROJECT, task_name="premise_regularisation_summary", task_type=Task.TaskTypes.qc,
                 tags=["premise_regularisation", "summary"], reuse_last_task_id=False, auto_connect_frameworks={"matplotlib": False})
task.connect({**RECIPE, "seeds": SEEDS, "rule": {"min_gain": MIN_GAIN, "max_top1_drop": MAX_TOP1_DROP, "max_mca_drop": MAX_MCA_DROP, "tie": TIE},
              "split_sha256": split_hash, "evaluated_splits": list(SPLITS)})
task.get_logger().report_table("regularisation arms (validation)", "val", iteration=0, table_plot=show.reset_index())
task.get_logger().report_matplotlib_figure("composite vs control", "val", figure=fig, iteration=0, report_interactive=False)
task.upload_artifact("selected_config", artifact_object=f"{REPO}/configs/premise_regularisation.json")
task.upload_artifact("results", artifact_object=results_path)
task.close()
config

## What happens next

- **Something passes.** Copy the selected configuration into the final comparison: 3 to 5 reporting seeds (42, 123, 7, plus two more) on the test set for
  the control and the selected arm, same offline plus online setup, so the gain (or its absence) is measured on seeds the tuning never saw.
- **Nothing passes.** The premise arm is already at its local optimum for these knobs; that is a finding, and the control is what gets reported.
- **Final model.** After the arm is settled, the model is retrained on train plus validation with the epoch count fixed at the mean best epoch
  stored in `configs/premise_regularisation.json`. The offline build-up is regenerated from the train plus validation clips for that run, into its own folder, and the
  test set is evaluated once, with no selection.